<div class="alert alert-success py-2 px-3 mb-2"
     style="font-family: 'Montserrat', sans-serif; font-size: 16px;">

<style>
.alert code {
    background: transparent !important;
    color: inherit !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13px !important;
}
</style>

<strong>Lab 01 – Hub Auth and Access</strong>

<strong>Notes:</strong> <code>notes/01-environment-and-auth.md</code> ·

<strong>Goal:</strong> Prove which token does what, which token wins, and how to stay safe.

<strong>Needs:</strong> <code>HF_READ_TOKEN</code> and <code>HF_WRITE_TOKEN</code> in the repository-root <code>.env</code> file.

</div>


In [8]:
import os, subprocess
from pathlib import Path
from dotenv import load_dotenv

import huggingface_hub
from huggingface_hub import HfApi, login, logout, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError, GatedRepoError

load_dotenv()

READ_TOKEN  = os.environ["HF_READ_TOKEN"]
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]
FINE_TOKEN  = os.getenv("HF_FINEGRAINED_TOKEN")      # optional

os.environ.pop("HF_TOKEN", None)                     # start with no env token
# Removes HF_TOKEN from the current process environment if it exists.
# The second argument, None, prevents a KeyError if HF_TOKEN was never set.
# This is useful because some Hugging Face libraries automatically use HF_TOKEN; removing it ensures your explicit tokens are used instead.



USER         = HfApi(token=READ_TOKEN).whoami()["name"]
SCRATCH_REPO = f"{USER}/bert-imdb-sentiment"
RESULTS_DIR  = Path.cwd().parent / "results"

def role_of(token):
    """Token role as the Hub sees it. token=None uses env, then saved token."""
    info = HfApi(token=token).whoami()
    return info.get("auth", {}).get("accessToken", {}).get("role")

def hf(*args):
    r = subprocess.run(["hf", *args], capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print("user:", USER)

user: akshayredekar74


In [9]:
print("read token role:", role_of(READ_TOKEN))

read token role: read


In [10]:
print("write token role:", role_of(WRITE_TOKEN))

write token role: write


In [11]:
print(hf("auth", "whoami"))

✓ Logged in
  user: akshayredekar74


In [12]:
print("huggingface_hub:", huggingface_hub.__version__)
print(hf("version"))

huggingface_hub: 1.11.0
1.11.0


In [31]:
# ── Lab 01 - Hub Auth and Access ──
# Notes : notes/01-environment-and-auth.md
# Result: results/01-auth-matrix.md
# Goal  : prove which token does what, which wins, how to stay safe
# Needs : HF_READ_TOKEN and HF_WRITE_TOKEN in the repo-root .env
# Warning: this lab logs in and out, so it changes your saved token

# ── imports ──
import os
from pathlib import Path
from dotenv import load_dotenv
import huggingface_hub
from huggingface_hub import HfApi, login, logout, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError, GatedRepoError

# ── load tokens from .env ──
load_dotenv()
READ_TOKEN = os.environ["HF_READ_TOKEN"]
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]

# ── start clean: no HF_TOKEN ──
os.environ.pop("HF_TOKEN", None)

# ── helper: what role does a token have? ──
# token=None means "use whatever the library finds"
def role_of(token):
    info = HfApi(token=token).whoami()
    return info["auth"]["accessToken"]["role"]

# ── who are we? ──
USER = HfApi(token=READ_TOKEN).whoami()["name"]
print("Logged in as:", USER)

Logged in as: akshayredekar74


In [32]:
# ── 1. install check ──
# Both should print a version number

print("huggingface_hub:", huggingface_hub.__version__)
!hf version

huggingface_hub: 1.11.0


1.11.0


In [33]:
# ── 2. token roles ──
# Ask the Hub what each token is (tokens are never printed)

print("read token role :", role_of(READ_TOKEN))
print("write token role:", role_of(WRITE_TOKEN))

read token role : read
write token role: write


In [34]:
# ── 3. login method 1: CLI ──
# Save the read token under the name "read"
!hf auth login --token $READ_TOKEN --token-name read

# ── who am I now? ──
!hf auth whoami
print("active role:", role_of(None))     # expect: read

Usage: hf auth login [OPTIONS]
Try 'hf auth login -h' for help.

Error: No such option: --token-name

Available options for 'hf auth login':
  --token TEXT                   A User Access Token generated from https://huggingface.co/settings/tokens.
  --add-to-git-credential / --no-add-to-git-credential Save to git credential helper. Useful only if you plan to run git commands directly.  [default: no-add-to-git-credential]
  --force / --no-force           Force re-login even if already logged in.  [default: no-force]
  -h, --help                     Show this message and exit.

Run 'hf auth login --help' for full details.

Did you mean: --token?
✓ Logged in
  user: akshayredekar74
active role: write


In [35]:
# ── 4. login method 2: login() ──
# Replaces the saved token with the write token

login(token=WRITE_TOKEN)
print("active role:", role_of(None))     # expect: write

active role: write


In [37]:
# ── 5. login method 3: env variable ──
# The library reads ONLY the name HF_TOKEN by itself
# HF_READ_TOKEN / HF_WRITE_TOKEN are just our own names

# ── saved token is write, env var should win ──
os.environ["HF_TOKEN"] = READ_TOKEN
print("active role:", role_of(None))     # expect: read

# ── remove it again ──
# os.environ.pop("HF_TOKEN")

active role: read


In [38]:
# ── 6. which token wins? ──
# Order: token= argument > HF_TOKEN env var > saved token

# ── setup: saved = write, env = read ──
login(token=WRITE_TOKEN)
os.environ["HF_TOKEN"] = READ_TOKEN

# ── test each layer ──
by_argument = role_of(WRITE_TOKEN)       # token= argument
env_and_disk = role_of(None)             # env + disk both present
os.environ.pop("HF_TOKEN")
disk_only = role_of(None)                # only saved token left

# ── results ──
print("token= argument:", by_argument)   # expect: write
print("env + disk     :", env_and_disk)  # expect: read
print("disk only      :", disk_only)     # expect: write

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


token= argument: write
env + disk     : read
disk only      : write


In [39]:
# ── 7. auth matrix ──
# Same upload, different token
# Read token must fail with 403, write token must succeed

# ── setup: private scratch repo ──
SCRATCH_REPO = f"{USER}/auth-matrix-scratch"
write_api = HfApi(token=WRITE_TOKEN)
write_api.create_repo(SCRATCH_REPO, private=True, exist_ok=True)

# ── helper: try an upload ──
def try_upload(name, token):
    try:
        HfApi(token=token).upload_file(
            path_or_fileobj=b"auth probe",
            path_in_repo=f"probe-{name}.txt",
            repo_id=SCRATCH_REPO,
        )
        return "OK"
    except HfHubHTTPError as e:
        return f"FAIL {e.response.status_code}"

# ── run the test ──
read_result = try_upload("read", READ_TOKEN)
write_result = try_upload("write", WRITE_TOKEN)
print("read token :", read_result)       # expect: FAIL 403
print("write token:", write_result)      # expect: OK

# ── clean up ──
write_api.delete_repo(SCRATCH_REPO)
print("deleted", SCRATCH_REPO)

read token : FAIL 403
write token: OK
deleted akshayredekar74/auth-matrix-scratch


In [42]:
# ── 8. save results ──
# Writes results/01-auth-matrix.md

# ── make sure the folder exists ──
results_dir = Path.cwd().parent / "results"
results_dir.mkdir(exist_ok=True)

# ── build the markdown text ──
text = f"""# 01 - Auth Matrix

huggingface_hub {huggingface_hub.__version__}

## Test 1: Can each token upload a file?

| Token | Role | Upload result |
|---|---|---|
| read | {role_of(READ_TOKEN)} | {read_result} |
| write | {role_of(WRITE_TOKEN)} | {write_result} |

A read token is blocked (403). A write token is allowed (OK).

## Test 2: Which token wins when several are present?

Setup: the token saved on my PC is a write token, and the
HF_TOKEN environment variable holds a read token.

| What I did | Tokens available | Role that was used |
|---|---|---|
| Passed token=... in code | the token I passed | {by_argument} |
| Passed nothing, HF_TOKEN is set | HF_TOKEN + saved token | {env_and_disk} |
| Passed nothing, HF_TOKEN is not set | saved token only | {disk_only} |

Rule: token=... wins, then HF_TOKEN, then the saved token.
"""

# ── write the file ──
(results_dir / "01-auth-matrix.md").write_text(text)
print("saved:", results_dir / "01-auth-matrix.md")

saved: /teamspace/studios/this_studio/All-about-finetuning/finetuning/00_platform/results/01-auth-matrix.md


In [43]:
# ── 9. gated repos ──
# A read token is enough, but first accept the terms
# on the model page. Change GATED_REPO to test another model.

GATED_REPO = "google/gemma-2-2b"

# ── try to download one small file ──
try:
    hf_hub_download(GATED_REPO, "config.json", token=READ_TOKEN)
    print("Access OK - the read token is enough")
except GatedRepoError:
    print("Gated: accept the terms on the model page, then run again")
except HfHubHTTPError as e:
    print("Other error:", e.response.status_code)

Gated: accept the terms on the model page, then run again


In [44]:
# ── 9. gated repos ──
# A read token is enough, but first accept the terms
# on the model page. Change GATED_REPO to test another model.

GATED_REPO = "google/gemma-2-2b"

# ── try to download one small file ──
try:
    hf_hub_download(GATED_REPO, "config.json", token=READ_TOKEN)
    print("Access OK - the read token is enough")
except GatedRepoError:
    print("Gated: accept the terms on the model page, then run again")
except HfHubHTTPError as e:
    print("Other error:", e.response.status_code)

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

Access OK - the read token is enough


In [49]:
# ── 3. login method 1: CLI ──
# The saved name comes from the token's name on huggingface.co
!hf auth login --token $READ_TOKEN

# ── who am I now? ──
!hf auth whoami
print("active role:", role_of(None))     # expect: read

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `READ_TOKEN` has been saved to /teamspace/studios/this_studio/.cache/huggingface/stored_tokens
Your token has been saved to /teamspace/studios/this_studio/.cache/huggingface/token
Login successful.
The current active token is: `READ_TOKEN`
✓ Logged in
  user: akshayredekar74
active role: read


In [53]:
# ── 10. Multiple saved tokens ─────────────────────────────────────────
import warnings

from IPython.display import display, Markdown
from huggingface_hub import whoami

warnings.filterwarnings("ignore")

# Save and check the read token
!hf auth login --token $READ_TOKEN
read_role = whoami()["auth"]["accessToken"]["role"]

# Replace it with the write token
!hf auth login --token $WRITE_TOKEN
write_role = whoami()["auth"]["accessToken"]["role"]

display(Markdown(f"""
**Multiple saved tokens**

- Active role after logging in with `READ_TOKEN`: **{read_role}**
- Active role after logging in with `WRITE_TOKEN`: **{write_role}**

The read token should show `read`.
The write token should show `write`.
"""))

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `READ_TOKEN` has been saved to /teamspace/studios/this_studio/.cache/huggingface/stored_tokens
Your token has been saved to /teamspace/studios/this_studio/.cache/huggingface/token
Login successful.
The current active token is: `READ_TOKEN`
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `WRITE_TOKEN` has been saved to /teamspace/studios/this_studio/.cache/huggingface/stored_tokens
Your token has been saved to /teamspace/studios/this_studio/.cache/huggingface/token
Login successful.
The current active token is: `WRITE_


**Multiple saved tokens**

- Active role after logging in with `READ_TOKEN`: **read**
- Active role after logging in with `WRITE_TOKEN`: **write**

The read token should show `read`.
The write token should show `write`.


In [54]:
# Read the whoami output. You should be able to
# explain every field you see.

!hf auth whoami --format json

{"user": "akshayredekar74", "orgs": null, "endpoint": null}


In [56]:
# ── 12. The logout trap ───────────────────────────────────────────────
import os
import warnings

from IPython.display import display, Markdown
from huggingface_hub import logout, whoami

warnings.filterwarnings("ignore")

# 1. Set the read token in the environment
os.environ["HF_TOKEN"] = READ_TOKEN

# 2. Remove the environment variable first
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)

# 3. Now logout can remove the saved token
logout()

display(Markdown("""

`logout()` removes the token saved on your machine.

It does **not** log you out if `HF_TOKEN` or `HUGGING_FACE_HUB_TOKEN`
is still set in the environment.

To fully log out:

1. Remove `HF_TOKEN`
2. Remove `HUGGING_FACE_HUB_TOKEN`
3. Call `logout()`
"""))

Not logged in!




`logout()` removes the token saved on your machine.

It does **not** log you out if `HF_TOKEN` or `HUGGING_FACE_HUB_TOKEN`
is still set in the environment.

To fully log out:

1. Remove `HF_TOKEN`
2. Remove `HUGGING_FACE_HUB_TOKEN`
3. Call `logout()`
